# Step 12: Cox proportional hazards models

Needs a machine with a lot of memory (the main specification has 5.7 M authors; a full run takes several hours).
Input: `cox_data.csv` (output of `step11_cox_dataprep.ipynb` (step 11)). Set `DATA_PATH` / `OUT_DIR` below and run all cells. All results are written as tidy CSV files to `OUT_DIR`;
`step13_publication_graphs.ipynb` (step 13) builds all figures and tables from these files.

**What is estimated** (for every specification in `SPECS`):

| model | covariates | baseline hazard |
|---|---|---|
| M0 | gender | common |
| M1 | gender | stratified by entry year |
| M2 | gender + female share + topic size (log) + topic maturity + first-year papers (2, 3+) | stratified by entry year |
| M3 | M2 + gender × every entry variable | stratified by entry year |
| M4 | gender + restricted cubic splines (4 knots) of female share, topic size, maturity + first-year papers | stratified by entry year |
| M5 | M4 + gender × every spline/term | stratified by entry year |
| M2_field, M3_field | as M2/M3 | stratified by entry year × field of entry topic |
| M2_cohort | as M2 but the gender effect is estimated separately for entry periods | stratified by entry year |

**Specifications** (`SPECS`): all models are estimated on the full analysis sample (entry 1950–2018, authors active in
≥ 2 distinct years, dropout = 2 years without publication). The other specifications are robustness checks:

| spec | sample / definition | models |
|---|---|---|
| `main` | entry 1950–2018 | M0–M5, field strata, cohort-specific gender effects |
| `check_2002` | entry 2002–2018 (full first names in MEDLINE) | M0–M3, field strata, cohort |
| `check_pre2002` | entry 1950–2001 | M0–M3, field strata, cohort |
| `check_gap3` | dropout = 3 years without publication | M0–M3, field strata, cohort |
| `check_all_authors` | incl. authors active in one year only | M0–M3, field strata, cohort |
| `check_fs_loo` | as `main`, but female share at entry **without the focal author** (leave-one-out) | M0–M3, field strata, cohort |
| `check_pkg_2002` | entry 2002–2018, only authors whose gender comes from their PKG forename (no recovered names); restricted to 2002+ because before 2002 PKG forenames are mostly initials, so almost only authors whose careers reach beyond 2002 are classified from them | M0–M3, field strata, cohort |

`check_fs_loo`: `female_share_entry` includes the author her/himself, so a woman mechanically raises her own female share
(most in small topic-years), which could feed the gender × female-share interaction. This spec uses the leave-one-out share
instead (spec option `fs_var`, default `female_share_entry`).

**Why the entry year is a stratum and not a covariate:** `begin_year` violates proportional hazards
strongly (Schoenfeld tests of models with a linear `begin_year` term) and is highly correlated with female share at entry (r ≈ 0.7). Stratifying gives every
entry cohort its own baseline hazard, so all other effects are estimated *within* cohorts. A linear `begin_year`
term leaves much of the time trend in the female-share coefficient.

**Coding (all continuous variables centred at the sample median):**
`fs10` = female share at entry in units of 10 percentage points; `size_log` = log(1 + topic size);
`maturity` = topic maturity in years; `fyp_2`, `fyp_3plus` = 2 / 3+ papers in first year (reference: 1).
Hence in interaction models `gender_f` is the female/male hazard ratio for an author entering at median conditions
with one first-year paper, and `f_x_<var>` is the change of that ratio per unit of `<var>`.

## 0. Configuration

In [ ]:
import os
import time
import json
import warnings
import gc
import numpy as np
import pandas as pd
from scipy import stats
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
DATA_PATH = f'{BASE_DIR}/data/derived/cox_data.csv'   # output of step 11
OUT_DIR = f'{BASE_DIR}/results'                       # one sub-folder per specification

# analysis specifications: entry cohorts, drop one-year authors?, dropout gap, which model set
SPECS = {
    'main':              dict(entry=(1950, 2018), o1y=True,  gap=2, models='full'),    # main analysis (all models)
    'check_2002':        dict(entry=(2002, 2018), o1y=True,  gap=2, models='linear'),  # cohorts with full first names
    'check_pre2002':     dict(entry=(1950, 2001), o1y=True,  gap=2, models='linear'),
    'check_gap3':        dict(entry=(1950, 2018), o1y=True,  gap=3, models='linear'),  # dropout = 3 years without publication
    'check_all_authors': dict(entry=(1950, 2018), o1y=False, gap=2, models='linear'),  # incl. one-year authors
    'check_pkg_2002':    dict(entry=(2002, 2018), o1y=True,  gap=2, models='linear', gender_source='pkg'),  # only gender from PKG forenames
    'check_fs_loo':      dict(entry=(1950, 2018), o1y=True,  gap=2, models='linear',
                             fs_var='female_share_entry_loo'),  # female share at entry without the focal author
}
RUN_SPECS = list(SPECS)          # e.g. ['main'] for a quick run, or a subset to split the specs over several sessions
PH_SAMPLE_N = 100_000            # random subsample for Schoenfeld tests (tests on millions of rows reject everything)
SPLINE_KNOTS_PCT = [5, 35, 65, 95]   # Harrell's default knot positions for 4 knots
SEED = 20261013

os.makedirs(OUT_DIR, exist_ok=True)
warnings.filterwarnings('ignore', category=FutureWarning)
t0 = time.time()
def log(msg):
    print(f'[{(time.time() - t0) / 60:6.1f} min] {msg}', flush=True)

## 1. Load data

In [ ]:
usecols = ['AID', 'gender_f', 'begin_year', 'last_pubyear', 'one_year', 'topics_num', 'field',
           'first_year_papers', 'topic_size_entry', 'topic_maturity_entry',
           'female_share_entry', 'female_share_entry_loo',
           'event_2y', 'duration_2y', 'event_3y', 'duration_3y', 'gender_source']
dtypes = {'gender_f': 'int8', 'begin_year': 'int16', 'last_pubyear': 'int16', 'one_year': 'int8',
          'first_year_papers': 'int16', 'topic_size_entry': 'int32', 'field': 'category',
          'event_2y': 'int8', 'event_3y': 'int8', 'duration_2y': 'int16', 'duration_3y': 'int16',
          'gender_source': 'category'}
raw = pd.read_csv(DATA_PATH, usecols=usecols, dtype=dtypes)
log(f'loaded {len(raw):,} authors')
raw.describe().T

## 2. Helper functions

### 2.1 Sample and variable construction

In [ ]:
def rcs_basis(x, knots):
    """Restricted cubic spline basis (Harrell 2015, ch. 2.4.5): linear term + len(knots)-2 non-linear terms,
    linear beyond the outer knots. Non-linear terms are scaled by (t_k - t_1)^2 for numerical stability."""
    x = np.asarray(x, dtype=float)
    k = np.asarray(knots, dtype=float)
    K = len(k)
    cube = lambda u: np.maximum(u, 0.0) ** 3
    cols = [x]
    for j in range(K - 2):
        term = (cube(x - k[j])
                - cube(x - k[K - 2]) * (k[K - 1] - k[j]) / (k[K - 1] - k[K - 2])
                + cube(x - k[K - 1]) * (k[K - 2] - k[j]) / (k[K - 1] - k[K - 2]))
        cols.append(term / (k[K - 1] - k[0]) ** 2)
    return np.column_stack(cols)


SPLINE_VARS = {  # model name stem -> raw variable on the scale used in the model
    'fs':       'fs_raw10',
    'size':     'size_log_raw',
    'maturity': 'maturity_raw',
}


def make_sample(raw, spec):
    """Select the analysis sample of a specification and construct all model variables."""
    lo, hi = spec['entry']
    gap = spec['gap']
    fs_var = spec.get('fs_var', 'female_share_entry')   # female-share measure used in the models
    keep = ['AID', 'gender_f', 'begin_year', 'one_year', 'field', 'first_year_papers', 'topic_size_entry',
            'topic_maturity_entry', 'female_share_entry', 'female_share_entry_loo',
            f'event_{gap}y', f'duration_{gap}y']
    sel = (raw.begin_year >= lo) & (raw.begin_year <= hi)
    if spec.get('gender_source'):
        sel &= raw.gender_source == spec['gender_source']
    d = raw.loc[sel, keep].copy()
    if spec['o1y']:
        d = d[d.one_year == 0]
    d['field'] = d['field'].astype(str)                 # plain strings: safe as lifelines strata
    d['event'] = d[f'event_{gap}y'].astype(int)
    d['duration'] = d[f'duration_{gap}y'].astype(float)
    d = d[d.duration > 0]
    d = d[d[fs_var].notna()]                            # LOO share undefined if the author is the only classified one

    # raw (uncentred) scales
    d['fs_raw10'] = d[fs_var] * 10                       # 1 unit = 10 percentage points
    d['size_log_raw'] = np.log1p(d.topic_size_entry)
    d['maturity_raw'] = d.topic_maturity_entry.astype(float)

    # centred versions (sample medians) used in all linear models
    ref = {}
    for new, src in [('fs10', 'fs_raw10'), ('size_log', 'size_log_raw'), ('maturity', 'maturity_raw')]:
        ref[new] = float(d[src].median())
        d[new] = d[src] - ref[new]
    d['fyp_2'] = (d.first_year_papers == 2).astype(int)
    d['fyp_3plus'] = (d.first_year_papers >= 3).astype(int)

    # explicit gender interactions (no formula parser needed -> transparent and fast)
    for v in ['fs10', 'size_log', 'maturity', 'fyp_2', 'fyp_3plus']:
        d[f'f_x_{v}'] = d.gender_f * d[v]

    # spline bases, centred at the median so that gender_f = gap at median conditions
    knots = {}
    for stem, src in (SPLINE_VARS.items() if spec['models'] == 'full' else []):
        kn = np.unique(np.percentile(d[src], SPLINE_KNOTS_PCT))   # duplicates possible for integer variables
        knots[stem] = kn.tolist()
        ref_val = float(d[src].median())
        B = rcs_basis(d[src], kn) - rcs_basis([ref_val], kn)
        for j in range(B.shape[1]):
            d[f'{stem}_rcs{j}'] = B[:, j]
            d[f'f_x_{stem}_rcs{j}'] = d.gender_f * B[:, j]

    # entry periods for gender-by-cohort models (4-year periods from 2002, decades before)
    edges = [2002, 2006, 2010, 2014] if lo >= 2002 else list(range(lo, hi + 1, 10))
    labels = [f'{a}-{b - 1}' for a, b in zip(edges, edges[1:])] + [f'{edges[-1]}-{hi}']
    d['period'] = pd.cut(d.begin_year, bins=edges + [hi + 1], right=False, labels=labels).astype(str)
    for p in labels:
        d[f'f_period_{p}'] = d.gender_f * (d.period == p).astype(int)

    meta = dict(reference_values=ref, knots=knots)
    return d.reset_index(drop=True), meta

### 2.2 Model definitions

In [ ]:
LIN = ['fs10', 'size_log', 'maturity', 'fyp_2', 'fyp_3plus']
def spline_cols(d):
    return [c for c in d.columns if '_rcs' in c and not c.startswith('f_x_')]


def model_set(spec_name, spec, d):
    """Return {model_name: (covariates, strata)}."""
    st = ['begin_year']
    m = {
        'M0': (['gender_f'], None),
        'M1': (['gender_f'], st),
        'M2': (['gender_f'] + LIN, st),
        'M3': (['gender_f'] + LIN + [f'f_x_{v}' for v in LIN], st),
        'M2_field': (['gender_f'] + LIN, st + ['field']),
        'M3_field': (['gender_f'] + LIN + [f'f_x_{v}' for v in LIN], st + ['field']),
        'M2_cohort': ([c for c in d.columns if c.startswith('f_period_')] + LIN, st),
    }
    if spec['models'] == 'full':
        SPL = spline_cols(d)
        m['M4'] = (['gender_f'] + SPL + ['fyp_2', 'fyp_3plus'], st)
        m['M5'] = (['gender_f'] + SPL + ['fyp_2', 'fyp_3plus']
                   + [f'f_x_{c}' for c in SPL] + ['f_x_fyp_2', 'f_x_fyp_3plus'], st)
    return m

### 2.3 Fitting and tidy output

In [ ]:
def fit_cox(d, covs, strata):
    cols = covs + ['duration', 'event'] + (strata or [])
    data = d[cols].dropna()
    cph = CoxPHFitter()
    cph.fit(data, duration_col='duration', event_col='event', strata=strata)
    cph.n_obs_, cph.n_events_ = len(data), int(data['event'].sum())   # stored for the model table
    return cph, data


def tidy(cph, spec_name, model_name):
    s = cph.summary.reset_index().rename(columns={'covariate': 'term', 'index': 'term'})
    out = pd.DataFrame({
        'spec': spec_name, 'model': model_name, 'term': s['term'],
        'coef': s['coef'], 'se': s['se(coef)'],
        'HR': s['exp(coef)'], 'HR_low': s['exp(coef) lower 95%'], 'HR_high': s['exp(coef) upper 95%'],
        'z': s['z'], 'p': s['p']})
    return out


def model_stats(cph, spec_name, model_name, covs, strata):
    try:
        cidx = cph.concordance_index_
    except Exception:
        cidx = np.nan
    return dict(spec=spec_name, model=model_name, n=cph.n_obs_, events=cph.n_events_,
                log_likelihood=cph.log_likelihood_, aic_partial=cph.AIC_partial_, concordance=cidx,
                n_params=len(cph.params_), strata='+'.join(strata or []), covariates=' + '.join(covs))


def vcov_long(cph, spec_name, model_name):
    v = cph.variance_matrix_.copy()
    v.index.name, v.columns.name = 'term_1', 'term_2'
    out = v.stack().rename('cov').reset_index()
    out.insert(0, 'model', model_name)
    out.insert(0, 'spec', spec_name)
    return out


def lin_comb(cph, weights):
    """HR and 95% CI of a linear combination sum_k w_k * beta_k (uses the full covariance matrix)."""
    names = list(weights)
    w = np.array([weights[n] for n in names], dtype=float)
    b = cph.params_.loc[names].values
    V = cph.variance_matrix_.loc[names, names].values
    est = float(w @ b)
    se = float(np.sqrt(w @ V @ w))
    return est, se, np.exp(est), np.exp(est - 1.959964 * se), np.exp(est + 1.959964 * se)

### 2.4 Derived quantities: gender gap profiles, spline curves, cohort-specific gender effects

In [ ]:
PROFILE_VARS = {'fs10': 'fs_raw10', 'size_log': 'size_log_raw', 'maturity': 'maturity_raw'}


def linear_profiles(cph, d, meta, spec_name, model_name):
    """For linear interaction models: female/male HR (gender gap) and the effect of each variable for men and women,
    evaluated at the 1st..99th percentile of the variable (all other variables at the reference)."""
    rows = []
    for v, src in PROFILE_VARS.items():
        if v not in cph.params_.index:
            continue
        grid = np.percentile(d[src], np.linspace(1, 99, 50))
        for x in grid:
            dx = x - meta['reference_values'][v]
            parts = {
                'gap':   {'gender_f': 1.0, f'f_x_{v}': dx},
                'men':   {v: dx},
                'women': {v: dx, f'f_x_{v}': dx},     # women relative to women at the reference value
            }
            for kind, w in parts.items():
                w = {k: val for k, val in w.items() if k in cph.params_.index}
                est, se, hr, lo_, hi_ = lin_comb(cph, w)
                rows.append(dict(spec=spec_name, model=model_name, variable=v, x_raw=x, kind=kind,
                                 HR=hr, HR_low=lo_, HR_high=hi_))
    return pd.DataFrame(rows)


def spline_curves(cph, d, meta, spec_name, model_name):
    """Predicted HR curves from spline models (M4: overall; M5: men, women and gender gap),
    relative to the median of the variable; grid = 1st..99th percentile."""
    rows = []
    has_int = any(c.startswith('f_x_') for c in cph.params_.index)
    for stem, src in SPLINE_VARS.items():
        kn = meta['knots'][stem]
        ref_val = float(d[src].median())
        grid = np.percentile(d[src], np.linspace(1, 99, 60))
        B = rcs_basis(grid, kn) - rcs_basis([ref_val], kn)
        cols = [f'{stem}_rcs{j}' for j in range(B.shape[1])]
        for i, x in enumerate(grid):
            if has_int:
                kinds = {
                    'men':   {c: B[i, j] for j, c in enumerate(cols)},
                    'women': {**{c: B[i, j] for j, c in enumerate(cols)},
                              **{f'f_x_{c}': B[i, j] for j, c in enumerate(cols)}},
                    'gap':   {'gender_f': 1.0, **{f'f_x_{c}': B[i, j] for j, c in enumerate(cols)}},
                }
            else:
                kinds = {'overall': {c: B[i, j] for j, c in enumerate(cols)}}
            for kind, w in kinds.items():
                est, se, hr, lo_, hi_ = lin_comb(cph, w)
                rows.append(dict(spec=spec_name, model=model_name, variable=stem, x_raw=x, kind=kind,
                                 HR=hr, HR_low=lo_, HR_high=hi_))
    return pd.DataFrame(rows)


def lrt(cph_small, cph_big, spec_name, name):
    stat = 2 * (cph_big.log_likelihood_ - cph_small.log_likelihood_)
    df_ = len(cph_big.params_) - len(cph_small.params_)
    return dict(spec=spec_name, comparison=name, lr_stat=stat, df=df_, p=stats.chi2.sf(stat, df_))


def descriptives(d, spec_name):
    v = ['female_share_entry', 'topic_size_entry', 'topic_maturity_entry', 'first_year_papers',
         'begin_year', 'duration', 'event']
    g = d.groupby('gender_f')[v].agg(['mean', 'std', 'median'])
    g.columns = [f'{a}_{b}' for a, b in g.columns]
    g['n'] = d.groupby('gender_f').size()
    g = g.reset_index().assign(spec=spec_name)
    return g

## 3. Run all specifications

Results are written to `OUT_DIR/<spec>/` after every model, so an interrupted run keeps everything finished so far,
and specifications can be run in separate sessions (set `RUN_SPECS`).

In [ ]:
def save(results, spec_dir):
    os.makedirs(spec_dir, exist_ok=True)
    for k, parts in results.items():
        if not parts:
            continue
        if k == 'meta':
            json.dump(parts, open(f'{spec_dir}/sample_meta.json', 'w'), indent=1)
        elif k in ('stats', 'lrt'):
            pd.DataFrame(parts).to_csv(f'{spec_dir}/{k}.csv', index=False)
        else:
            pd.concat(parts, ignore_index=True).to_csv(f'{spec_dir}/{k}.csv', index=False)

for spec_name in RUN_SPECS:
    spec = SPECS[spec_name]
    spec_dir = f'{OUT_DIR}/{spec_name}'          # one folder per specification -> specs can be run in separate sessions
    results = {k: [] for k in ['coef', 'stats', 'vcov', 'profiles', 'splines', 'lrt', 'ph', 'desc', 'meta']}
    d, meta = make_sample(raw, spec)
    meta.update(spec=spec_name, fs_var=spec.get('fs_var', 'female_share_entry'), n=len(d), events=int(d.event.sum()), entry=list(spec['entry']),
                o1y=spec['o1y'], gap=spec['gap'])
    results['meta'].append(meta)
    results['desc'].append(descriptives(d, spec_name))
    log(f'=== {spec_name}: {len(d):,} authors, {d.event.sum():,} dropouts')

    fitted = {}
    for model_name, (covs, strata) in model_set(spec_name, spec, d).items():
        cph, data = fit_cox(d, covs, strata)
        del data
        fitted[model_name] = cph
        results['coef'].append(tidy(cph, spec_name, model_name))
        results['stats'].append(model_stats(cph, spec_name, model_name, covs, strata))
        results['vcov'].append(vcov_long(cph, spec_name, model_name))
        if model_name in ('M3', 'M3_field'):
            results['profiles'].append(linear_profiles(cph, d, meta, spec_name, model_name))
        if model_name in ('M4', 'M5'):
            results['splines'].append(spline_curves(cph, d, meta, spec_name, model_name))
        g = cph.params_.get('gender_f', np.nan)
        log(f'{spec_name} {model_name}: HR(gender_f) = {np.exp(g):.3f}' if pd.notna(g) else f'{spec_name} {model_name} done')
        save(results, spec_dir)

    for small, big in [('M2', 'M3'), ('M2', 'M4'), ('M4', 'M5'), ('M2_field', 'M3_field')]:
        if small in fitted and big in fitted:
            results['lrt'].append(lrt(fitted[small], fitted[big], spec_name, f'{small} vs {big}'))

    # proportional-hazards diagnostics on a random subsample (same specification as the full model)
    if spec_name in ('main', 'check_2002'):
        sub = d.sample(n=min(PH_SAMPLE_N, len(d)), random_state=SEED)
        for model_name in ('M2', 'M3'):
            covs, strata = model_set(spec_name, spec, d)[model_name]
            cph_s, data_s = fit_cox(sub, covs, strata)
            ph = proportional_hazard_test(cph_s, data_s, time_transform='rank')
            t = ph.summary.reset_index().rename(columns={'index': 'term', 'level_0': 'term'})
            t.insert(0, 'model', model_name)
            t.insert(0, 'spec', spec_name)
            results['ph'].append(t)
    save(results, spec_dir)
    del d, fitted
    gc.collect()

log('all done')

## 4. Quick look at the key results

In [ ]:
import glob
def read_all(name):
    files = sorted(glob.glob(f'{OUT_DIR}/*/{name}.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

coef = read_all('coef')
key = coef[coef.term.isin(['gender_f', 'fs10', 'f_x_fs10'])]
key.pivot_table(index=['spec', 'model'], columns='term', values='HR').round(3)

In [ ]:
read_all('stats')[['spec', 'model', 'n', 'events', 'log_likelihood', 'aic_partial', 'concordance']]

In [ ]:
coef[(coef.model == 'M2_cohort') & coef.term.str.startswith('f_period_')][['spec', 'term', 'HR', 'HR_low', 'HR_high']]